## Problem 01: Smart Thermostat & Invariant Encapsulation Engine

### Class Design and Encapsulation

The `SmartThermostat` class represents a smart thermostat device. It stores the device ID, current temperature, and target temperature as private attributes.

The `current_temp` and `target_temp` attributes are accessed through `@property` and `@setter`. This provides encapsulation and allows temperature validation before changing the internal state.

The target temperature is restricted to the range **10.0°C to 35.0°C**. If an invalid target temperature is provided, the update is rejected and an error message is displayed.

The class also provides:
- `to_fahrenheit()` to convert Celsius to Fahrenheit.
- `get_mode()` to determine whether the thermostat is in `HEATING`, `COOLING`, or `IDLE` mode.
- `__str__()` to display the thermostat status in the required format.

This design keeps the thermostat's internal state protected while providing controlled access through properties and methods.

In [3]:
class SmartThermostat:
    """Represents a smart thermostat with validated temperature settings."""

    def __init__(
        self,
        device_id: str,
        current_temp: float,
        target_temp: float
    ) -> None:
        self.__device_id = device_id
        self.__current_temp = current_temp
        self.__target_temp = target_temp

    @property
    def current_temp(self) -> float:
        """Return the current temperature in Celsius."""
        return self.__current_temp

    @current_temp.setter
    def current_temp(self, temp: float) -> None:
        """Update the current temperature."""
        self.__current_temp = temp
        print(f"Current Updated: {temp:.2f}C")

    @property
    def target_temp(self) -> float:
        """Return the target temperature in Celsius."""
        return self.__target_temp

    @target_temp.setter
    def target_temp(self, temp: float) -> None:
        """Update target temperature after validation."""
        if 10.0 <= temp <= 35.0:
            self.__target_temp = temp
            print(f"Target Updated: {temp:.2f}C")
        else:
            print("Error: Invalid Target Temperature")

    def to_fahrenheit(self, celsius: float) -> float:
        """Convert Celsius temperature to Fahrenheit."""
        return round((celsius * 9 / 5) + 32, 2)

    def get_mode(self) -> str:
        """Return the current thermostat operating mode."""
        if self.__current_temp < self.__target_temp:
            return "HEATING"
        if self.__current_temp > self.__target_temp:
            return "COOLING"
        return "IDLE"

    def __str__(self) -> str:
        """Return the formatted thermostat status."""
        current_f = self.to_fahrenheit(self.__current_temp)
        return (
            f"Thermostat[{self.__device_id}] "
            f"Current: {self.__current_temp:.2f}C ({current_f:.2f}F) | "
            f"Target: {self.__target_temp:.2f}C | "
            f"Mode: {self.get_mode()}"
        )

In [4]:
thermostat = SmartThermostat("TH101", 18.5, 22.0)

print(thermostat)

thermostat.target_temp = 25.0
thermostat.target_temp = 40.0

thermostat.current_temp = 25.0

print(thermostat)

Thermostat[TH101] Current: 18.50C (65.30F) | Target: 22.00C | Mode: HEATING
Target Updated: 25.00C
Error: Invalid Target Temperature
Current Updated: 25.00C
Thermostat[TH101] Current: 25.00C (77.00F) | Target: 25.00C | Mode: IDLE


## Problem 02: Library Media Hierarchy & Polymorphic Overdue Engine

### Class Design and Inheritance

The `LibraryItem` class is the base class for all library media items. It stores the common attributes `item_id`, `title`, and `max_days`, and provides a default `calculate_fine()` method.

Three subclasses are derived from `LibraryItem`:

- `Book` has a maximum loan period of 14 days and charges $1.00 per overdue day.
- `Audiobook` has a maximum loan period of 7 days, stores `duration_hours`, and charges $1.50 per overdue day.
- `DVD` has a maximum loan period of 3 days, stores `director`, and charges $2.00 per overdue day with an additional $5.00 surcharge when overdue by more than 5 days.

Each subclass overrides `calculate_fine()` according to its own rules. The `CirculationDesk` manages a heterogeneous list of `LibraryItem` objects and calculates fines polymorphically without checking the object's type using `isinstance`.

This design demonstrates inheritance, `super().__init__()`, method overriding, and runtime polymorphism.

In [5]:
class LibraryItem:
    """Base class for library media items."""

    def __init__(self, item_id: str, title: str, max_days: int) -> None:
        self.item_id = item_id
        self.title = title
        self.max_days = max_days

    def calculate_fine(self, days_late: int) -> float:
        """Calculate the default overdue fine."""
        return 0.50 * days_late


class Book(LibraryItem):
    """Represents a library book."""

    def __init__(self, item_id: str, title: str) -> None:
        super().__init__(item_id, title, 14)

    def calculate_fine(self, days_late: int) -> float:
        """Calculate the book overdue fine."""
        return 1.00 * days_late


class Audiobook(LibraryItem):
    """Represents an audiobook."""

    def __init__(
        self,
        item_id: str,
        title: str,
        duration_hours: float
    ) -> None:
        super().__init__(item_id, title, 7)
        self.duration_hours = duration_hours

    def calculate_fine(self, days_late: int) -> float:
        """Calculate the audiobook overdue fine."""
        return 1.50 * days_late


class DVD(LibraryItem):
    """Represents a DVD."""

    def __init__(
        self,
        item_id: str,
        title: str,
        director: str
    ) -> None:
        super().__init__(item_id, title, 3)
        self.director = director

    def calculate_fine(self, days_late: int) -> float:
        """Calculate the DVD overdue fine."""
        fine = 2.00 * days_late

        if days_late > 5:
            fine += 5.00

        return fine


class CirculationDesk:
    """Manages library items and calculates total overdue fines."""

    def __init__(self) -> None:
        self.items = []

    def add_item(self, item: LibraryItem) -> None:
        """Add a library item to the circulation desk."""
        self.items.append(item)

    def total_fines(self, overdue_days: list[int]) -> float:
        """Calculate total fines polymorphically."""
        return sum(
            item.calculate_fine(days)
            for item, days in zip(self.items, overdue_days)
        )

In [6]:
book = Book("B101", "PythonDSA")
audiobook = Audiobook("A201", "CleanCode", 8.5)
dvd = DVD("D301", "Inception", "Nolan")

desk = CirculationDesk()

desk.add_item(book)
desk.add_item(audiobook)
desk.add_item(dvd)

overdue_days = [5, 4, 6]

for item, days in zip(desk.items, overdue_days):
    fine = item.calculate_fine(days)

    if isinstance(item, Book):
        print(f"{item.item_id} {item.title} (Book): Fine = ${fine:.2f}")
    elif isinstance(item, Audiobook):
        print(
            f"{item.item_id} {item.title} "
            f"(Audiobook, {item.duration_hours} hrs): Fine = ${fine:.2f}"
        )
    elif isinstance(item, DVD):
        print(
            f"{item.item_id} {item.title} "
            f"(DVD, Dir: {item.director}): Fine = ${fine:.2f}"
        )

print(f"Total Outstanding Fines: ${desk.total_fines(overdue_days):.2f}")

B101 PythonDSA (Book): Fine = $5.00
A201 CleanCode (Audiobook, 8.5 hrs): Fine = $6.00
D301 Inception (DVD, Dir: Nolan): Fine = $17.00
Total Outstanding Fines: $28.00


## Problem 03: Multi-Channel Notification Gateway

### Class Design and Abstraction

The `NotificationService` class is an Abstract Base Class (ABC) that defines the common interface for all notification services. It declares the abstract methods `send()` and `get_cost()`.

Three concrete services inherit from this abstract class:

- `EmailService` sends messages through email with a fixed cost of $0.01.
- `SMSService` sends messages through SMS, limits messages to 160 characters, and charges $0.05 per 160-character segment.
- `WebhookService` sends notifications through a webhook endpoint with no cost.

The `AlertDispatcher` aggregates multiple notification services and broadcasts an alert polymorphically. It calls the same methods on each service without depending on the specific implementation.

This design demonstrates abstraction, abstract methods, interface segregation, and polymorphic dispatch while keeping the alert dispatcher independent of individual notification providers.

In [7]:
from abc import ABC, abstractmethod
import math


class NotificationService(ABC):
    """Abstract base class for notification services."""

    @abstractmethod
    def send(self, recipient: str, message: str) -> str:
        """Send a notification to the recipient."""
        pass

    @abstractmethod
    def get_cost(self, message: str) -> float:
        """Return the cost of sending the message."""
        pass


class EmailService(NotificationService):
    """Notification service for email."""

    def __init__(self, sender_email: str) -> None:
        self.sender_email = sender_email

    def send(self, recipient: str, message: str) -> str:
        """Send an email notification."""
        return (
            f"Email sent to {recipient} via "
            f"{self.sender_email}: {message}"
        )

    def get_cost(self, message: str) -> float:
        """Return the fixed email cost."""
        return 0.01


class SMSService(NotificationService):
    """Notification service for SMS."""

    def __init__(self, phone_number: str) -> None:
        self.phone_number = phone_number

    def send(self, recipient: str, message: str) -> str:
        """Send an SMS notification with a 160-character limit."""
        if len(message) > 160:
            message = message[:157] + "..."

        return f"SMS sent to {recipient}: {message}"

    def get_cost(self, message: str) -> float:
        """Calculate SMS cost based on 160-character segments."""
        segments = math.ceil(len(message) / 160)
        return 0.05 * segments


class WebhookService(NotificationService):
    """Notification service for webhook endpoints."""

    def __init__(self, endpoint_url: str) -> None:
        self.endpoint_url = endpoint_url

    def send(self, recipient: str, message: str) -> str:
        """Send a webhook notification."""
        return (
            f"POST payload to {self.endpoint_url} "
            f"for {recipient}: {message}"
        )

    def get_cost(self, message: str) -> float:
        """Return the free webhook cost."""
        return 0.00


class AlertDispatcher:
    """Broadcasts alerts through multiple notification services."""

    def __init__(self, services: list[NotificationService]) -> None:
        self.services = services

    def broadcast(self, recipient: str, message: str) -> float:
        """Broadcast an alert and return the total dispatch cost."""
        total_cost = 0.0

        for service in self.services:
            print(service.send(recipient, message))
            cost = service.get_cost(message)
            print(f"Cost: ${cost:.2f}")
            total_cost += cost

        return total_cost

In [8]:
services = [
    EmailService("alerts@cloud.com"),
    SMSService("+1999888777"),
    WebhookService("https://hooks.slack.com/services/alert")
]

dispatcher = AlertDispatcher(services)

recipient = "AdminUser"
message = "CRITICAL: High CPU utilization detected on server node 04."

total_cost = dispatcher.broadcast(recipient, message)

print(f"Total Broadcast Cost: ${total_cost:.2f}")

Email sent to AdminUser via alerts@cloud.com: CRITICAL: High CPU utilization detected on server node 04.
Cost: $0.01
SMS sent to AdminUser: CRITICAL: High CPU utilization detected on server node 04.
Cost: $0.05
POST payload to https://hooks.slack.com/services/alert for AdminUser: CRITICAL: High CPU utilization detected on server node 04.
Cost: $0.00
Total Broadcast Cost: $0.06


## Problem 04: Exact Rational Fraction Engine & Operator Overloading

### Class Design and Encapsulation

The `Rational` class represents a fraction using a numerator and denominator. The numerator and denominator are stored as private attributes to maintain encapsulation.

The constructor enforces important invariants:
- The denominator cannot be zero.
- The denominator is always kept positive.
- The fraction is reduced to its simplest form using the greatest common divisor (GCD).

Operator overloading is used to perform arithmetic directly between `Rational` objects. The class overloads `__add__`, `__sub__`, `__mul__`, and `__truediv__` for arithmetic operations.

Comparison operators such as `__eq__` and `__lt__` allow rational numbers to be compared without converting them to floating-point values. The `__str__` method provides a readable fraction representation, while `__float__` converts the rational number to a floating-point value.

This design provides exact arithmetic and avoids floating-point precision errors.

In [9]:
from math import gcd


class Rational:
    """Represents an exact rational number."""

    def __init__(self, numerator: int, denominator: int) -> None:
        if denominator == 0:
            raise ZeroDivisionError("Denominator cannot be zero.")

        if denominator < 0:
            numerator = -numerator
            denominator = -denominator

        common = gcd(abs(numerator), denominator)

        self.__numerator = numerator // common
        self.__denominator = denominator // common

    def __add__(self, other: "Rational") -> "Rational":
        """Add two rational numbers."""
        return Rational(
            self.__numerator * other.__denominator
            + other.__numerator * self.__denominator,
            self.__denominator * other.__denominator
        )

    def __sub__(self, other: "Rational") -> "Rational":
        """Subtract two rational numbers."""
        return Rational(
            self.__numerator * other.__denominator
            - other.__numerator * self.__denominator,
            self.__denominator * other.__denominator
        )

    def __mul__(self, other: "Rational") -> "Rational":
        """Multiply two rational numbers."""
        return Rational(
            self.__numerator * other.__numerator,
            self.__denominator * other.__denominator
        )

    def __truediv__(self, other: "Rational") -> "Rational":
        """Divide two rational numbers."""
        if other.__numerator == 0:
            raise ZeroDivisionError("Cannot divide by zero.")

        return Rational(
            self.__numerator * other.__denominator,
            self.__denominator * other.__numerator
        )

    def __eq__(self, other: object) -> bool:
        """Compare two rational numbers for equality."""
        if not isinstance(other, Rational):
            return False

        return (
            self.__numerator * other.__denominator
            == other.__numerator * self.__denominator
        )

    def __lt__(self, other: "Rational") -> bool:
        """Compare two rational numbers using less-than."""
        return (
            self.__numerator * other.__denominator
            < other.__numerator * self.__denominator
        )

    def __str__(self) -> str:
        """Return the simplified fraction as a string."""
        if self.__denominator == 1:
            return str(self.__numerator)

        return f"{self.__numerator}/{self.__denominator}"

    def __float__(self) -> float:
        """Convert the rational number to a float."""
        return self.__numerator / self.__denominator

In [10]:
f1 = Rational(1, 3)
f2 = Rational(1, 6)

print(f"f1 = {f1}, f2 = {f2}")
print(f"f1 + f2 = {f1 + f2} (float: {float(f1 + f2):.4f})")
print(f"f1 - f2 = {f1 - f2}")
print(f"f1 * f2 = {f1 * f2}")
print(f"f1 / f2 = {f1 / f2}")
print(f"f1 > f2: {f2 < f1}")
print(f"f1 == f2: {f1 == f2}")

f1 = 1/3, f2 = 1/6
f1 + f2 = 1/2 (float: 0.5000)
f1 - f2 = 1/6
f1 * f2 = 1/18
f1 / f2 = 2
f1 > f2: True
f1 == f2: False


## Problem 05: Multi-Entity E-Commerce Fulfillment & State Machine Engine

### Class Design and Composition

The system consists of three main domain classes: `Product`, `Customer`, and `Order`.

The `Product` class stores product information such as product ID, name, price, and stock. The `Customer` class stores customer information and wallet balance. The `Order` class connects a customer with selected products and quantities and maintains the order status.

The order follows a finite state machine:

CREATED → PAID → SHIPPED → DELIVERED

An order can also be cancelled from the `CREATED` or `PAID` state. Invalid state transitions are rejected to protect the order lifecycle.

The `ECommerceEngine` aggregates products, customers, and orders. It validates stock while creating an order, reserves stock, handles payments, restores stock when required, processes refunds during cancellation, and manages valid order-state transitions.

This design demonstrates multi-class composition, domain aggregation, state machine transitions, and invariant guarding.

In [11]:
class Product:
    """Represents a product available for purchase."""

    def __init__(
        self,
        product_id: str,
        name: str,
        price: float,
        stock: int
    ) -> None:
        self.product_id = product_id
        self.name = name
        self.price = price
        self.stock = stock


class Customer:
    """Represents a customer with a wallet balance."""

    def __init__(
        self,
        customer_id: str,
        name: str,
        wallet_balance: float
    ) -> None:
        self.customer_id = customer_id
        self.name = name
        self.wallet_balance = wallet_balance


class Order:
    """Represents an order and its lifecycle state."""

    def __init__(
        self,
        order_id: str,
        customer: Customer,
        items: dict[Product, int]
    ) -> None:
        self.order_id = order_id
        self.customer = customer
        self.items = items
        self.status = "CREATED"

    def total(self) -> float:
        """Calculate the total order value."""
        return sum(
            product.price * quantity
            for product, quantity in self.items.items()
        )


class ECommerceEngine:
    """Manages products, customers, orders, and order transitions."""

    def __init__(self) -> None:
        self.products: dict[str, Product] = {}
        self.customers: dict[str, Customer] = {}
        self.orders: dict[str, Order] = {}

    def add_product(self, product: Product) -> None:
        """Add a product to the system."""
        self.products[product.product_id] = product

    def add_customer(self, customer: Customer) -> None:
        """Add a customer to the system."""
        self.customers[customer.customer_id] = customer

    def create_order(
        self,
        order_id: str,
        customer_id: str,
        items: dict[str, int]
    ) -> str:
        """Validate stock, reserve products, and create an order."""
        customer = self.customers[customer_id]
        order_items: dict[Product, int] = {}

        for product_id, quantity in items.items():
            product = self.products[product_id]

            if product.stock < quantity:
                return "Error: Insufficient Stock"

            order_items[product] = quantity

        for product, quantity in order_items.items():
            product.stock -= quantity

        order = Order(order_id, customer, order_items)
        self.orders[order_id] = order

        return f"Order {order_id} Created: Total = ${order.total():.2f}"

    def pay_order(self, order_id: str) -> str:
        """Process payment and transition the order to PAID."""
        order = self.orders[order_id]

        if order.status != "CREATED":
            return f"Error: Cannot pay in {order.status} state."

        total = order.total()

        if order.customer.wallet_balance < total:
            for product, quantity in order.items.items():
                product.stock += quantity

            order.status = "CANCELLED"
            return "Payment Failed: Insufficient Balance"

        order.customer.wallet_balance -= total
        order.status = "PAID"

        return "Order " + order_id + " Paid Successfully."

    def ship_order(self, order_id: str) -> str:
        """Transition a paid order to SHIPPED."""
        order = self.orders[order_id]

        if order.status != "PAID":
            return f"Error: Cannot ship in {order.status} state."

        order.status = "SHIPPED"
        return f"Order {order_id} Status: SHIPPED"

    def deliver_order(self, order_id: str) -> str:
        """Transition a shipped order to DELIVERED."""
        order = self.orders[order_id]

        if order.status != "SHIPPED":
            return f"Error: Cannot deliver in {order.status} state."

        order.status = "DELIVERED"
        return f"Order {order_id} Status: DELIVERED"

    def cancel_order(self, order_id: str) -> str:
        """Cancel an order and restore stock and balance when required."""
        order = self.orders[order_id]

        if order.status not in ("CREATED", "PAID"):
            return f"Error: Cannot cancel in {order.status} state."

        if order.status == "PAID":
            order.customer.wallet_balance += order.total()

        for product, quantity in order.items.items():
            product.stock += quantity

        order.status = "CANCELLED"

        return f"Order {order_id} Cancelled."

In [12]:
engine = ECommerceEngine()

engine.add_product(Product("P101", "Laptop", 1000.00, 5))
engine.add_product(Product("P102", "Mouse", 50.00, 10))

engine.add_customer(Customer("C01", "Alice", 1500.00))

items = {
    "P101": 1,
    "P102": 2
}

print(engine.create_order("O1", "C01", items))
print(engine.pay_order("O1"))

order = engine.orders["O1"]
print(f"{order.customer.name} Remaining Balance: ${order.customer.wallet_balance:.2f}")

print(engine.ship_order("O1"))
print(engine.deliver_order("O1"))
print(engine.cancel_order("O1"))

Order O1 Created: Total = $1100.00
Order O1 Paid Successfully.
Alice Remaining Balance: $400.00
Order O1 Status: SHIPPED
Order O1 Status: DELIVERED
Error: Cannot cancel in DELIVERED state.
